# Notebook 03 — Cluster Analysis

**Goal:** Identify user segments using KMeans and HDBSCAN on the behavioural feature matrix. Select the best model via silhouette/elbow analysis, evaluate cluster quality, and characterise each segment.

**Inputs:** `data/processed/user_features.parquet`

**Outputs:**
- `data/processed/cluster_labels.parquet` — user IDs with cluster assignments
- `outputs/figures/elbow.html` — KMeans model selection chart
- `outputs/figures/umap_clusters.html` — 2D UMAP scatter
- `outputs/figures/cluster_heatmap.html` — feature profile heatmap

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

import pandas as pd
import numpy as np

Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
from src.data.loader import load_config

cfg = load_config('../configs/config.yaml')
cluster_cfg = cfg['clustering']

feature_matrix = pd.read_parquet('../data/processed/user_features.parquet')
print(f'Feature matrix: {feature_matrix.shape[0]} users × {feature_matrix.shape[1]} features')

## 1. KMeans — Elbow & Silhouette Analysis

Sweep k from 3 to 10. PCA is applied first to remove multicollinearity.

In [ ]:
from src.clustering.pipeline import run_clustering_pipeline
from src.clustering.evaluation import elbow_data
from src.visualization.plots import plot_elbow, save_figure

# Run KMeans sweep
kmeans_result = run_clustering_pipeline(
    feature_matrix=feature_matrix,
    algorithm='kmeans',
    use_pca=True,
    use_umap_viz=True,
    config=cluster_cfg,
)

print(f'Best k: {kmeans_result.n_clusters}')
print(f'Silhouette: {kmeans_result.silhouette:.4f}')
print(f'Davies-Bouldin: {kmeans_result.davies_bouldin:.4f}')
print(f'Calinski-Harabasz: {kmeans_result.calinski_harabasz:.1f}')

In [ ]:
# Full k sweep for elbow diagnostics — always runs k=3..10 regardless of final k choice
from src.clustering.pipeline import preprocess, reduce_with_pca, cluster_kmeans

X_scaled, feature_names = preprocess(feature_matrix)
X_pca, pca_obj = reduce_with_pca(X_scaled)

k_min = cluster_cfg['kmeans']['n_clusters_range'][0]  # 3
k_max = cluster_cfg['kmeans']['n_clusters_range'][1]  # 10

_, auto_best_k, sil_scores, inertias = cluster_kmeans(
    X_pca,
    k_range=(k_min, k_max),
    n_init=cluster_cfg['kmeans']['n_init'],
    random_state=cluster_cfg['kmeans']['random_state'],
)
print(f'Metric-optimal k: {auto_best_k}  (highest silhouette across k={k_min}-{k_max})')

elbow_df = elbow_data(inertias, sil_scores)
fig_elbow = plot_elbow(elbow_df)
save_figure(fig_elbow, '../outputs/figures/elbow')
fig_elbow.show()

In [ ]:
# ── Manual k override ────────────────────────────────────────────────────────────────────────────
# Silhouette metrics prefer k=3, but 3 segments lacks the behavioural
# granularity needed for meaningful personalisation recommendations.
# k=5 is chosen to produce richer, more interpretable listener archetypes.
CHOSEN_K = 5

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from src.clustering.pipeline import ClusterResult, embed_umap

km_final = KMeans(
    n_clusters=CHOSEN_K,
    n_init=cluster_cfg['kmeans']['n_init'],
    random_state=cluster_cfg['kmeans']['random_state'],
)
labels_k5 = km_final.fit_predict(X_pca)

sil_k5 = silhouette_score(X_pca, labels_k5)
db_k5  = davies_bouldin_score(X_pca, labels_k5)
ch_k5  = calinski_harabasz_score(X_pca, labels_k5)

print(f'k={CHOSEN_K} — Silhouette: {sil_k5:.4f} | Davies-Bouldin: {db_k5:.4f} | Calinski-Harabasz: {ch_k5:.1f}')

umap_coords_k5 = embed_umap(X_pca)

manual_kmeans_result = ClusterResult(
    labels=labels_k5,
    algorithm='kmeans',
    n_clusters=CHOSEN_K,
    silhouette=sil_k5,
    davies_bouldin=db_k5,
    calinski_harabasz=ch_k5,
    feature_matrix_scaled=X_scaled,
    feature_names=feature_names,
    umap_coords=umap_coords_k5,
    pca_variance_explained=pca_obj.explained_variance_ratio_,
    model=km_final,
)

## 2. HDBSCAN — Density-Based Clustering

In [ ]:
hdbscan_result = run_clustering_pipeline(
    feature_matrix=feature_matrix,
    algorithm='hdbscan',
    use_pca=True,
    use_umap_viz=True,
    config=cluster_cfg,
)

print(f'HDBSCAN clusters: {hdbscan_result.n_clusters}')
print(f'Noise points: {(hdbscan_result.labels == -1).sum()}')
print(f'Silhouette: {hdbscan_result.silhouette:.4f}')

## 3. Model Selection

Compare KMeans (best k) vs HDBSCAN on key metrics.

In [ ]:
comparison = pd.DataFrame([
    {
        'Algorithm': f'KMeans (k={kmeans_result.n_clusters})',
        'Clusters': kmeans_result.n_clusters,
        'Noise Points': 0,
        'Silhouette ↑': kmeans_result.silhouette,
        'Davies-Bouldin ↓': kmeans_result.davies_bouldin,
        'Calinski-Harabasz ↑': kmeans_result.calinski_harabasz,
    },
    {
        'Algorithm': 'HDBSCAN',
        'Clusters': hdbscan_result.n_clusters,
        'Noise Points': int((hdbscan_result.labels == -1).sum()),
        'Silhouette ↑': hdbscan_result.silhouette,
        'Davies-Bouldin ↓': hdbscan_result.davies_bouldin,
        'Calinski-Harabasz ↑': hdbscan_result.calinski_harabasz,
    },
])
comparison.set_index('Algorithm', inplace=True)
comparison.round(4)

In [ ]:
# Use manually chosen k=5 KMeans result as the best model.
# HDBSCAN is shown in the comparison table for reference only.
best_result = manual_kmeans_result
print(f'Using: KMeans k={CHOSEN_K} (manually selected for interpretability)')
print(f'Metric-optimal k was: {auto_best_k}')

## 4. Cluster Stability (Bootstrap Silhouette)

In [ ]:
from src.clustering.evaluation import bootstrap_silhouette

mean_sil, std_sil = bootstrap_silhouette(
    best_result.feature_matrix_scaled,
    best_result.labels,
    n_bootstrap=50,
)
print(f'Bootstrap silhouette: {mean_sil:.4f} ± {std_sil:.4f}')

## 5. Cluster Profiling

In [ ]:
from src.clustering.evaluation import summarise_clusters, label_clusters, feature_importance

cluster_summary = summarise_clusters(feature_matrix, best_result.labels)
cluster_names = label_clusters(cluster_summary, feature_matrix, best_result.labels)

print('Cluster labels:')
for cid, name in cluster_names.items():
    n = (best_result.labels == cid).sum()
    print(f'  Cluster {cid} ({n} users): {name}')

In [ ]:
# Feature importance across clusters
imp_df = feature_importance(feature_matrix, best_result.labels)
print('Top 10 most discriminating features:')
print(imp_df.head(10).to_string(index=False))

## 6. Save Cluster Labels

In [ ]:
labels_df = pd.DataFrame({
    'userid': feature_matrix.index,
    'cluster': best_result.labels,
    'cluster_name': [cluster_names.get(c, str(c)) for c in best_result.labels],
})

if best_result.umap_coords is not None:
    labels_df['umap_x'] = best_result.umap_coords[:, 0]
    labels_df['umap_y'] = best_result.umap_coords[:, 1]

labels_df.to_parquet('../data/processed/cluster_labels.parquet', index=False)
print(f'Cluster labels saved: {labels_df.shape}')
labels_df['cluster_name'].value_counts()

Proceed to **Notebook 04** for interactive visualisations and business insights.